# Skyscraper Race — playable notebook

Same **3 data structures + 3 algorithms** as `logic/` in the repo.
Code style matches the project: explicit loops, clear names, short comments.

| Kind | Piece | File in repo | Role |
|------|-------|--------------|------|
| DS | Hash table | `logic/data/materials.py` | material lookup |
| DS | List | `logic/algorithms/tower.py` | tower floors |
| DS | Min-heap | `logic/algorithms/challenges.py` | next challenge height |
| Algo | Recursion | `tower.load_above` | weight above a floor |
| Algo | Heap peek/pop | `challenges.run_challenges` | fire next challenge |
| Algo | Sequential search | `tower.find_weakest_floor_index` | find weakest floor |

Run all cells for the auto-demo. No Tk / GUI needed.

## Settings

Same knobs as `logic/data/constants.py`.

In [ ]:
# Balance knobs in one place (same as logic/data/constants.py).

TARGET_HEIGHT = 150
ROLLS_PER_TURN = 3

ROLL_MIN = 10000
ROLL_MAX = 99999

# Digit 1 → tier. A dict maps digits in O(1).
TIER_RANGES = {
    1: "junk", 2: "junk", 3: "junk",
    4: "common", 5: "common", 6: "common", 7: "common",
    8: "strong", 9: "strong",
}

UNITS_PER_AMOUNT = 150
CAPACITY_CONSTANT = 1200

NUM_CHALLENGES = 3
CHALLENGE_MIN_HEIGHT = 50
CHALLENGE_MAX_HEIGHT = 130
CHALLENGE_MIN_GAP = 25
EARTHQUAKE_DAMAGE = 0.06
WINDSTORM_DAMAGE = 0.10
DRUNK_DRIVER_DAMAGE = 0.15

print("ok")

## DS #1 — Hash table

Same as `logic/data/materials.py`.

Why: each roll needs material stats by `(tier, group)`. A dict gives average **O(1)** lookup.
Better than if/elif or scanning a list.

In [ ]:
# Key = (tier, group). Digit 3 of the roll picks group "a" (0–4) or "b" (5–9).
MATERIALS = {
    ("junk", "a"): {
        "name": "toothpicks",
        "weight": 0.5,
        "strength": 0.3,
    },
    ("junk", "b"): {
        "name": "rotten_wood",
        "weight": 0.8,
        "strength": 0.2,
    },
    ("common", "a"): {
        "name": "wood",
        "weight": 1.0,
        "strength": 1.0,
    },
    ("common", "b"): {
        "name": "brick",
        "weight": 2.0,
        "strength": 1.6,
    },
    ("strong", "a"): {
        "name": "steel",
        "weight": 2.5,
        "strength": 4.5,
    },
    ("strong", "b"): {
        "name": "concrete",
        "weight": 3.0,
        "strength": 3.0,
    },
}


def tier_for_digit(digit):
    # O(1) dict read — same idea as MATERIALS.
    return TIER_RANGES[digit]


def get_material(tier, material_digit):
    # Map digit → group, then one hash-table lookup for the stats row.
    if material_digit <= 4:
        group = "a"
    else:
        group = "b"

    key = (tier, group)
    material = MATERIALS[key]
    return material


def units_from_amount(amount):
    units = amount * UNITS_PER_AMOUNT
    return units


print(get_material("strong", 9))

## Rolls

Same as `logic/algorithms/roll.py`.

Why one number: easy to print, test, and replay. Decoding is **O(1)**.

In [ ]:
import random


def generate_roll():
    # Uniform 5-digit int; full_decode turns it into material + height.
    roll = random.randint(ROLL_MIN, ROLL_MAX)
    return roll


def decode_roll(roll):
    # Split digits: tier, amount, material pick, height (d4+d5, min 2).
    text = str(roll)
    if len(text) != 5:
        raise ValueError("roll must be 5 digits")

    digit1 = int(text[0])
    digit2 = int(text[1])
    digit3 = int(text[2])
    digit4 = int(text[3])
    digit5 = int(text[4])

    if digit2 == 0:
        amount = 10
    else:
        amount = digit2

    height = digit4 + digit5
    if height < 2:
        height = 2

    result = {
        "roll": roll,
        "tier": tier_for_digit(digit1),
        "amount": amount,
        "material_digit": digit3,
        "height": height,
    }
    return result


def full_decode(roll):
    # Attach hash-table material stats so Floor(...) is ready in one step.
    parts = decode_roll(roll)
    material = get_material(parts["tier"], parts["material_digit"])
    units = units_from_amount(parts["amount"])

    result = {
        "roll": roll,
        "tier": parts["tier"],
        "material_name": material["name"],
        "weight_per_unit": material["weight"],
        "capacity_mult": material["strength"],
        "units": units,
        "height": parts["height"],
        "image": material["name"],
    }
    return result


print(full_decode(generate_roll()))

## Floor

Same as `logic/data/floor.py`.

Plain data object — no linked-list pointers. Weight / capacity stay **O(1)**.

In [ ]:
class Floor:
    def __init__(self, roll_data):
        # Copy the decoded roll into fields the tower algorithms will use.
        self.material_name = roll_data["material_name"]
        self.tier = roll_data["tier"]
        self.height = roll_data["height"]
        self.units = roll_data["units"]
        self.weight_per_unit = roll_data["weight_per_unit"]
        self.strength = roll_data["capacity_mult"]
        self.health = 1.0  # challenges multiply this down

        if "image" in roll_data:
            self.image = roll_data["image"]
        else:
            self.image = self.material_name

    def get_weight(self):
        # O(1): mass used when summing load above a floor.
        weight = self.units * self.weight_per_unit
        return weight

    def get_max_load(self):
        # O(1): how much weight this floor can hold right now.
        max_load = (
            self.strength
            * self.height
            * CAPACITY_CONSTANT
            * self.health
        )
        return max_load

    def take_damage(self, percent):
        # Multiplicative damage; never below 0.
        new_health = self.health * (1.0 - percent)
        if new_health < 0.0:
            new_health = 0.0
        self.health = new_health

    def label(self):
        text = self.material_name + ":" + str(self.height) + "m"
        return text

## DS #2 list · Algo #1 recursion · Algo #3 sequential search

Same as `logic/algorithms/tower.py`.

- **List**: index `0` = ground. Place-by-slot is natural with indices.
- **Recursion (`load_above`)**: load(i) = weight(i+1) + load(i+1); top → `0`. **O(n)**.
- **Sequential search**: any floor can be weakest — walk once, keep the worst ratio.

In [ ]:
class Tower:
    def __init__(self):
        # Contiguous list of Floor objects; [0] is always the ground floor.
        self.floors = []

    def is_empty(self):
        if len(self.floors) == 0:
            return True
        return False

    def floor_count(self):
        count = len(self.floors)
        return count

    def get_height(self):
        # O(n) sum — n is tiny; clearer than caching.
        total = 0.0
        index = 0
        while index < len(self.floors):
            floor = self.floors[index]
            total = total + floor.height
            index = index + 1
        return total

    def floors_from_ground(self):
        # Copy so callers can iterate without mutating the tower.
        result = []
        index = 0
        while index < len(self.floors):
            result.append(self.floors[index])
            index = index + 1
        return result

    def insert_at(self, position, new_floor):
        if position < 0:
            raise IndexError("bad position")
        if position > len(self.floors):
            raise IndexError("bad position")
        self.floors.insert(position, new_floor)

    def remove_at(self, index):
        removed = self.floors.pop(index)
        return removed

    # --- ALGORITHM #1: recursion ---

    def load_above(self, index):
        """Total weight resting on floors[index].

        Why recursion: load(i) is weight(i+1) + load(i+1); base = top → 0.
        O(n) time / O(n) stack — must see each floor above.
        """
        # Base case: this is the top floor (nothing above it).
        if index >= len(self.floors) - 1:
            return 0.0

        next_index = index + 1
        next_floor = self.floors[next_index]
        weight_of_next = next_floor.get_weight()
        weight_above_next = self.load_above(next_index)
        total = weight_of_next + weight_above_next
        return total

    def hold_ratio(self, index):
        # capacity / load. Under 1.0 means this floor fails.
        load = self.load_above(index)
        if load == 0:
            return 999.0

        floor = self.floors[index]
        max_load = floor.get_max_load()
        ratio = max_load / load
        return ratio

    # --- ALGORITHM #3: sequential search ---

    def find_weakest_floor_index(self):
        """Return the index with the worst hold_ratio.

        Why sequential search: any floor can be weakest; no sorted key.
        Walk once, keep the minimum ratio — O(n).
        """
        if self.is_empty() is True:
            return None

        # Start by assuming floor 0 is the weakest.
        weakest_index = 0
        worst_ratio = self.hold_ratio(0)

        # Check every other floor one by one.
        index = 1
        while index < len(self.floors):
            ratio = self.hold_ratio(index)
            if ratio < worst_ratio:
                worst_ratio = ratio
                weakest_index = index
            index = index + 1

        return weakest_index

    def check_stable(self):
        if self.is_empty() is True:
            return 999.0, None

        weakest_index = self.find_weakest_floor_index()
        ratio = self.hold_ratio(weakest_index)
        return ratio, weakest_index

    def collapse(self, broken_index=None):
        """Drop the broken floor and everything above it."""
        if broken_index is None:
            ratio, broken_index = self.check_stable()
            if broken_index is None:
                return [], 0.0
            if ratio >= 1.0:
                return [], 0.0

        if broken_index < 0:
            return [], 0.0
        if broken_index >= len(self.floors):
            return [], 0.0

        # Keep everything below the break; drop the rest.
        fallen = []
        index = broken_index
        while index < len(self.floors):
            fallen.append(self.floors[index])
            index = index + 1

        kept = []
        index = 0
        while index < broken_index:
            kept.append(self.floors[index])
            index = index + 1
        self.floors = kept

        meters_lost = 0.0
        index = 0
        while index < len(fallen):
            meters_lost = meters_lost + fallen[index].height
            index = index + 1

        # Cascade: after a cut, another floor may now be overloaded.
        ratio, weakest_index = self.check_stable()
        if weakest_index is not None:
            if ratio < 1.0:
                more_fallen, more_meters = self.collapse(weakest_index)
                index = 0
                while index < len(more_fallen):
                    fallen.append(more_fallen[index])
                    index = index + 1
                meters_lost = meters_lost + more_meters

        return fallen, meters_lost


t = Tower()
t.insert_at(0, Floor(full_decode(47382)))
print("height", t.get_height())

## DS #3 min-heap · Algo #2 peek/pop

Same as `logic/algorithms/challenges.py`.

Why a heap: next challenge is the **smallest** remaining height.
Peek **O(1)**, pop **O(log n)** — better than scanning an unsorted list each place.

In [ ]:
import heapq

EVENT_NAMES = ("earthquake", "windstorm", "drunk_driver")
DAMAGE = {
    "earthquake": EARTHQUAKE_DAMAGE,
    "windstorm": WINDSTORM_DAMAGE,
    "drunk_driver": DRUNK_DRIVER_DAMAGE,
}


def make_challenge_heights():
    # Place heights low→high with a minimum gap so events are not stacked.
    heights = []
    lowest_allowed = CHALLENGE_MIN_HEIGHT
    left = NUM_CHALLENGES

    while left > 0:
        highest_allowed = CHALLENGE_MAX_HEIGHT - CHALLENGE_MIN_GAP * (left - 1)
        if lowest_allowed > highest_allowed:
            break

        height = random.randint(lowest_allowed, highest_allowed)
        heights.append(height)
        lowest_allowed = height + CHALLENGE_MIN_GAP
        left = left - 1

    return heights

def make_heap(heights=None):
    # heapify is O(n) — cheaper than n inserts — and leaves min at index 0.
    if heights is None:
        heights = make_challenge_heights()

    heap = []
    index = 0
    while index < len(heights):
        heap.append(heights[index])
        index = index + 1

    heapq.heapify(heap)
    return heap

def apply_event(tower, event_name):
    # Apply one disaster. Ground hit uses index 0 — no search needed.
    floors = tower.floors_from_ground()
    if len(floors) == 0:
        return event_name + ": empty tower"

    damage = DAMAGE[event_name]
    percent = int(damage * 100)

    if event_name == "earthquake":
        index = 0
        while index < len(floors):
            floors[index].take_damage(damage)
            index = index + 1
        return "Earthquake -" + str(percent) + "% all floors"

    if event_name == "windstorm":
        # Hit up to the top 3 floors.
        start = len(floors) - 3
        if start < 0:
            start = 0
        index = start
        while index < len(floors):
            floors[index].take_damage(damage)
            index = index + 1
        return "Windstorm -" + str(percent) + "% top floors"

    # drunk_driver: always the ground floor
    floors[0].take_damage(damage)
    return "Drunk driver -" + str(percent) + "% ground"

def run_challenges(tower, challenge_heap):
    """While tower height ≥ next heap root, pop and apply.

    Why peek/pop: the heap root is the soonest challenge; after pop,
    the new root is the next soonest.
    """
    messages = []
    fired = []

    while len(challenge_heap) > 0:
        next_height = challenge_heap[0]  # peek: O(1)
        if next_height > tower.get_height():
            break

        height = heapq.heappop(challenge_heap)  # pop: O(log n)
        event_name = random.choice(EVENT_NAMES)
        message = "@" + str(height) + "m " + apply_event(tower, event_name)
        messages.append(message)
        fired.append(height)

    return messages, fired


print("heights", make_challenge_heights())

## Player + Game + Auto demo

Same as `logic/algorithms/player.py` and `logic/algorithms/game.py`.

Turn flow: roll → place → heap challenges → sequential stability → win.

In [ ]:
class Player:
    def __init__(self, name, rolls_per_turn=ROLLS_PER_TURN):
        self.name = name
        self.tower = Tower()
        self.rolls_per_turn = rolls_per_turn
        self.rolls_left = rolls_per_turn
        self.pending_roll = None
        self.won = False

        # Keep the raw list for debugging; heap is what gameplay uses.
        self.challenge_heights = make_challenge_heights()
        copy_for_heap = []
        index = 0
        while index < len(self.challenge_heights):
            copy_for_heap.append(self.challenge_heights[index])
            index = index + 1
        self.challenge_heap = make_heap(copy_for_heap)

    def reset_rolls(self):
        self.rolls_left = self.rolls_per_turn
        self.pending_roll = None

    def tower_height(self):
        height = self.tower.get_height()
        return height

    def floor_count(self):
        count = self.tower.floor_count()
        return count

    def next_challenge(self):
        # ALGORITHM #2 peek: min-heap root is the next (smallest) height.
        if len(self.challenge_heap) == 0:
            return None
        next_height = self.challenge_heap[0]
        return next_height


class Game:
    def __init__(self, name1="P1", name2="P2", enable_challenges=True):
        self.players = []
        self.players.append(Player(name1, ROLLS_PER_TURN))
        self.players.append(Player(name2, ROLLS_PER_TURN))

        self.current_index = 0
        self.phase = "roll"
        self.winner = None
        self.last_collapse = None
        self.enable_challenges = enable_challenges
        self.message = self._prompt_roll()

    def current_player(self):
        player = self.players[self.current_index]
        return player

    def _prompt_roll(self):
        player = self.current_player()
        text = player.name + "'s turn — roll a floor"
        return text

    def roll(self):
        if self.phase == "game_over":
            raise RuntimeError("Game over")
        if self.phase != "roll":
            raise RuntimeError("Place or scrap first")

        player = self.current_player()
        if player.rolls_left <= 0:
            raise RuntimeError("No rolls left")

        recipe = full_decode(generate_roll())
        player.pending_roll = recipe
        player.rolls_left = player.rolls_left - 1
        self.phase = "place"

        self.message = (
            "Rolled "
            + recipe["material_name"]
            + " "
            + str(recipe["height"])
            + "m"
        )
        return recipe

    def place_floor(self, position):
        player = self.current_player()
        if self.phase != "place":
            raise RuntimeError("Roll first")
        if player.pending_roll is None:
            raise RuntimeError("Roll first")

        recipe = player.pending_roll

        slot = position
        if slot < 0:
            slot = 0
        if slot > player.tower.floor_count():
            slot = player.tower.floor_count()

        player.tower.insert_at(slot, Floor(recipe))
        player.pending_roll = None
        self.last_collapse = None

        events = []
        if self.enable_challenges is True:
            # ALGORITHM #2: peek/pop challenges whose height ≤ tower height.
            events, _fired = run_challenges(
                player.tower,
                player.challenge_heap,
            )

        collapsed = False
        # ALGORITHM #1 + #3: recursive loads inside sequential weakest search.
        ratio, weakest_index = player.tower.check_stable()
        if weakest_index is not None:
            if ratio < 1.0:
                self.last_collapse = player.tower.collapse(weakest_index)
                collapsed = True

        height = player.tower_height()
        if collapsed is True:
            self.message = "Collapse! " + str(int(height)) + " m left"
        else:
            self.message = "Placed — " + str(int(height)) + " m"

        if events:
            self.message = self.message + " | " + "; ".join(events)

        if self.check_win() is True:
            return {"collapsed": collapsed, "won": True}

        if player.rolls_left > 0:
            self.phase = "roll"
        else:
            self.end_turn()

        return {"collapsed": collapsed, "won": False}

    def place_on_top(self):
        player = self.current_player()
        top_slot = player.tower.floor_count()
        result = self.place_floor(top_slot)
        return result

    def check_win(self):
        player = self.current_player()
        if player.tower_height() >= TARGET_HEIGHT:
            self.winner = player
            self.phase = "game_over"
            player.won = True
            self.message = player.name + " wins!"
            return True
        return False

    def end_turn(self):
        if self.phase == "game_over":
            return

        if self.current_index == 0:
            self.current_index = 1
        else:
            self.current_index = 0

        player = self.current_player()
        player.reset_rolls()
        self.phase = "roll"
        self.message = self._prompt_roll()


def auto_demo(seed=11, max_actions=40):
    random.seed(seed)
    g = Game()
    action = 0
    while action < max_actions:
        if g.phase == "game_over":
            break

        if g.phase == "roll":
            g.roll()

        g.place_on_top()
        print(g.message)

        player_index = 0
        while player_index < len(g.players):
            player = g.players[player_index]
            floors = player.tower.floors_from_ground()
            if len(floors) == 0:
                stack = "empty"
            else:
                labels = []
                floor_index = 0
                while floor_index < len(floors):
                    labels.append(floors[floor_index].label())
                    floor_index = floor_index + 1
                stack = " | ".join(labels)

            print(
                "  "
                + player.name
                + ": "
                + str(int(player.tower.get_height()))
                + "m ["
                + stack
                + "]"
            )
            player_index = player_index + 1

        print("-" * 40)
        action = action + 1

    if g.winner is not None:
        print("Winner:", g.winner.name)
    else:
        print("Winner: none yet")


auto_demo()